# 🛡️ SOC Threat Hunting & Behavioral Analysis
### Supervised alert triage + Unsupervised anomaly detection + SOC-aware threshold optimization

**Pipeline overview**

| Stage | What it does | Why it matters in a SOC |
|---|---|---|
| 1. Data adapter | Loads *UNSW-NB15* (train → test). Swap to *CERT* with one config switch | Same pipeline, different data |
| 2. Supervised engine | LightGBM + Random Forest, 3-tier triage (auto-close / analyst review / auto-escalate) | Cuts false positives and analyst workload |
| 3. Unsupervised engine | Isolation Forest + Autoencoder trained on **normal traffic only** | Catches novel / polymorphic behaviour that has no labels |
| 4. Threshold optimization | Cost-based, F-beta, precision-floor, and alert-budget strategies | A SOC cares about *analyst hours* and *missed attacks*, not accuracy |
| 5. Hybrid + zero-day test | Blend both engines; hold out whole attack families to test generalisation | Shows if the system can see attacks it never trained on |
| 6. Explainability + export | Per-alert top contributing features, saved models | Analysts need to know *why* an alert fired |

**Data discipline (important):**
`training-set` → split into *train* (fit models) and *validation* (early stopping + threshold tuning).
`testing-set` is touched **only once**, at the end, to report honest results.

## 0. Setup
Uncomment the install line if you are missing packages.

In [ ]:
# !pip install lightgbm scikit-learn pandas numpy matplotlib joblib

import warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # no automatic image/png output (avoids the VS Code renderer error)
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             confusion_matrix)
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.3})
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---- Robust figure display -------------------------------------------------
# Some VS Code setups cannot render 'image/png' outputs and print raw bytes instead.
# show() embeds the figure as an HTML <img> (always renderable) AND saves a PNG in ./figures/.
import io, base64
from IPython.display import display, HTML
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
_fig_counter = [0]
def show():
    _fig_counter[0] += 1
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    (FIG_DIR / f"fig_{_fig_counter[0]:02d}.png").write_bytes(buf.getvalue())
    b64 = base64.b64encode(buf.getvalue()).decode()
    display(HTML(f'<img src="data:image/png;base64,{b64}" style="max-width:100%"/>'))
    plt.close("all")

## 1. ⚙️ Configuration — *the only cell you need to edit to switch datasets*

* `DATASET` → `"UNSW_NB15"` now, `"CERT"` later.
* `DATA_DIR` → folder holding the CSV files.
* **SOC cost model** → how much a false alert costs vs. a missed attack (used for threshold tuning).

Every dataset adapter must return **two DataFrames (train, test)** with a binary `label` column (1 = malicious / insider, 0 = benign).
Optionally a category column (e.g. `attack_cat`) for per-family analysis.

In [ ]:
# ======================= USER SETTINGS =======================
DATASET  = "UNSW_NB15"          # <-- change to "CERT" later
DATA_DIR = Path(".")            # <-- folder containing the CSV files (UNSW) 
CERT_DIR = Path("./CERT")       # <-- folder containing logon.csv, device.csv, file.csv, email.csv, http.csv
CERT_ANSWERS = Path("./CERT/answers/insiders.csv")   # ground-truth insider list
CERT_VERSION = "4.2"            # CERT release (matches 'dataset' column in insiders.csv)

# ---- SOC economics (relative units; 1 unit ~ one analyst triaging one alert) ----
COST_FP = 1        # cost of a false alert (analyst time wasted)
COST_FN = 50       # cost of a missed attack (breach impact). Raise it for high-stakes SOCs.
F_BETA  = 2.0      # beta>1 favours recall

# ---- 3-tier triage policy (supervised engine) ----
MAX_MISSED_FRAC = 0.005   # auto-close zone may hide at most 0.5% of all attacks
MIN_PRECISION_ESC = 0.99  # auto-escalate zone must be >=99% precise

# ---- Unsupervised alert policy ----
PRECISION_FLOOR = 0.90    # for the 'precision floor' strategy
ALERT_BUDGET    = 0.05    # SOC can only look at 5% of events -> 'alert budget' strategy
AE_MAX_ROWS     = 60_000  # cap rows used to train the autoencoder (speed)
# =============================================================

### 1.1 Dataset adapters
Each adapter turns raw files into the common `(train_df, test_df)` contract.
The **UNSW-NB15** adapter is ready. The **CERT** adapter builds *user-day behaviour profiles* (logons, USB, files, e-mail, web) and is written for CERT r4.2-style files — adjust the column names if your release differs.

In [ ]:
# ------------------------- UNSW-NB15 -------------------------
def load_unsw():
    tr = pd.read_csv(DATA_DIR / "UNSW_NB15_training-set.csv", encoding="utf-8-sig")
    te = pd.read_csv(DATA_DIR / "UNSW_NB15_testing-set.csv",  encoding="utf-8-sig")
    return tr, te

def fe_unsw(df):
    """Network-flow feature engineering: ratios & totals often separate scans/DoS from normal flows."""
    d = df.copy()
    d["bytes_total"]   = d["sbytes"] + d["dbytes"]
    d["pkts_total"]    = d["spkts"] + d["dpkts"]
    d["byte_ratio"]    = (d["sbytes"] + 1) / (d["dbytes"] + 1)
    d["pkt_ratio"]     = (d["spkts"] + 1) / (d["dpkts"] + 1)
    d["bytes_per_pkt"] = d["bytes_total"] / d["pkts_total"].clip(lower=1)
    d["ttl_diff"]      = d["sttl"] - d["dttl"]
    d["load_ratio"]    = (d["sload"] + 1) / (d["dload"] + 1)
    return d


# ----------------------------- CERT -----------------------------
def _chunked_daily(name, usecols, feats, chunksize=500_000):
    """Stream a big CERT log and aggregate additive features per (user, day)."""
    out = []
    for ch in pd.read_csv(CERT_DIR / f"{name}.csv", usecols=usecols, chunksize=chunksize):
        ts = pd.to_datetime(ch["date"], format="%m/%d/%Y %H:%M:%S")
        g = pd.DataFrame({"user": ch["user"].values, "day": ts.dt.normalize().values})
        for k, f in feats.items():
            g[k] = np.asarray(f(ch, ts), dtype=float)
        out.append(g.groupby(["user", "day"]).sum())
    return pd.concat(out).groupby(level=[0, 1]).sum()

def fe_cert(df):
    """Per-user baselines: z-score of today's behaviour vs. the user's OWN past (expanding window, no leakage)."""
    df = df.sort_values(["user", "day"]).reset_index(drop=True)
    base = [c for c in df.columns if c not in ("user", "day", "label")]
    g = df.groupby("user")[base]
    mu = g.transform(lambda s: s.expanding(min_periods=5).mean().shift(1))
    sd = g.transform(lambda s: s.expanding(min_periods=5).std().shift(1))
    z = ((df[base] - mu) / (sd + 1e-6)).clip(-10, 10).fillna(0).add_suffix("_z")
    df = pd.concat([df, z], axis=1)
    df["dow"] = df["day"].dt.dayofweek
    return df

def load_cert(train_frac_days=0.7):
    """Builds a user-day table, labels insider days, and splits chronologically (train = past, test = future)."""
    after = lambda ts: (ts.dt.hour < 7) | (ts.dt.hour >= 19)
    parts = [
        _chunked_daily("logon", ["date", "user", "activity"], {
            "logon_n":  lambda c, t: c["activity"].eq("Logon"),
            "logoff_n": lambda c, t: c["activity"].eq("Logoff"),
            "logon_afterhours": lambda c, t: c["activity"].eq("Logon") & after(t)}),
        _chunked_daily("device", ["date", "user", "activity"], {
            "usb_connect": lambda c, t: c["activity"].eq("Connect"),
            "usb_afterhours": lambda c, t: c["activity"].eq("Connect") & after(t)}),
        _chunked_daily("file", ["date", "user"], {
            "file_n": lambda c, t: np.ones(len(c)),
            "file_afterhours": lambda c, t: after(t)}),
        _chunked_daily("email", ["date", "user", "to", "size", "attachments"], {
            "email_n": lambda c, t: np.ones(len(c)),
            "email_size": lambda c, t: c["size"],
            "email_attach": lambda c, t: c["attachments"],
            "email_external": lambda c, t: c["to"].fillna("").str.contains(r"@(?!dtaa\.com)", regex=True)}),
        _chunked_daily("http", ["date", "user"], {
            "http_n": lambda c, t: np.ones(len(c)),
            "http_afterhours": lambda c, t: after(t)}),
    ]
    daily = pd.concat(parts, axis=1).fillna(0).reset_index()

    ins = pd.read_csv(CERT_ANSWERS)
    ins = ins[ins["dataset"].astype(str) == CERT_VERSION].copy()
    ins["start"] = pd.to_datetime(ins["start"], format="%m/%d/%Y %H:%M:%S").dt.normalize()
    ins["end"]   = pd.to_datetime(ins["end"],   format="%m/%d/%Y %H:%M:%S").dt.normalize()
    daily = daily.merge(ins[["user", "start", "end"]], on="user", how="left")
    daily["label"] = ((daily["day"] >= daily["start"]) & (daily["day"] <= daily["end"])).astype(int)
    daily = fe_cert(daily.drop(columns=["start", "end"]).drop_duplicates(["user", "day"]))

    cut = daily["day"].quantile(train_frac_days)
    return daily[daily["day"] <= cut].copy(), daily[daily["day"] > cut].copy()


# ---------------------- Dataset registry ----------------------
DATASETS = {
    "UNSW_NB15": dict(loader=load_unsw, fe=fe_unsw, label_col="label", category_col="attack_cat",
                      drop_cols=["id"], entity="network flow"),
    "CERT":      dict(loader=load_cert, fe=None,    label_col="label", category_col=None,
                      drop_cols=["user", "day"], entity="user-day"),
}
cfg = DATASETS[DATASET]
LABEL, CAT_COL = cfg["label_col"], cfg["category_col"]
print(f"Active dataset: {DATASET}  ({cfg['entity']} records)")

## 2. Load data & quick EDA
**UNSW-NB15 note:** the official *training* and *testing* partitions have **different attack mixes** — this is deliberate and gives us a realistic *distribution-shift* test.

In [ ]:
train_raw, test_raw = cfg["loader"]()
if cfg["fe"] is not None:
    train_raw, test_raw = cfg["fe"](train_raw), cfg["fe"](test_raw)

print("train:", train_raw.shape, "| test:", test_raw.shape)
display(train_raw.head(3))

summary = pd.DataFrame({
    "train_rows": [len(train_raw)], "test_rows": [len(test_raw)],
    "train_attack_%": [100 * train_raw[LABEL].mean()], "test_attack_%": [100 * test_raw[LABEL].mean()]})
display(summary)

if CAT_COL:
    comp = pd.concat([train_raw[CAT_COL].value_counts(normalize=True).rename("train_%"),
                      test_raw[CAT_COL].value_counts(normalize=True).rename("test_%")], axis=1) * 100
    display(comp.sort_values("train_%", ascending=False))
    comp.plot.bar(figsize=(9, 4), title="Traffic mix: train vs test (distribution shift)")
    plt.ylabel("% of records"); plt.tight_layout(); show()

## 3. Feature preparation & data splits
* `FEATURES` = every column except IDs, label and attack category (**no label leakage**).
* **Supervised/IsolationForest** use an ordinal-encoded matrix (trees don't need scaling).
* **Autoencoder** uses signed-log + standardisation + one-hot (neural nets need scaled inputs).
* Both preprocessors are **fit on training rows only**.

In [ ]:
drop = set(cfg["drop_cols"] + [LABEL] + ([CAT_COL] if CAT_COL else []))
FEATURES = [c for c in train_raw.columns if c not in drop]
CAT_COLS = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(train_raw[c])]   # works for object AND pandas-3 "str" dtype
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]
print(f"{len(FEATURES)} features  ({len(NUM_COLS)} numeric, {len(CAT_COLS)} categorical: {CAT_COLS})")

# train -> (train, validation). Validation is used for early stopping + threshold tuning.
tr_df, va_df = train_test_split(train_raw, test_size=0.2, stratify=train_raw[LABEL], random_state=RANDOM_STATE)
tr_df, va_df, te_df = (d.reset_index(drop=True) for d in (tr_df, va_df, test_raw))
ytr, yva, yte = (d[LABEL].astype(int).values for d in (tr_df, va_df, te_df))

# --- preprocessor A: ordinal-encoded (LightGBM / RF / IsolationForest) ---
sup_pre = ColumnTransformer(
    [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS)],
    remainder="passthrough", verbose_feature_names_out=False)
sup_pre.set_output(transform="pandas")
Xtr = sup_pre.fit_transform(tr_df[FEATURES]);  Xva = sup_pre.transform(va_df[FEATURES]);  Xte = sup_pre.transform(te_df[FEATURES])

# --- preprocessor B: scaled + one-hot (Autoencoder) ---
def signed_log1p(x): return np.sign(x) * np.log1p(np.abs(x))
ae_pre = ColumnTransformer([
    ("num", Pipeline([("log", FunctionTransformer(signed_log1p)), ("sc", StandardScaler())]), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS)], sparse_threshold=0)
ae_pre.fit(tr_df.loc[ytr == 0, FEATURES])           # fit on NORMAL rows only
to_ae = lambda df: np.clip(ae_pre.transform(df[FEATURES]), -5, 5)

print("Splits  ->", {"train": len(tr_df), "val": len(va_df), "test": len(te_df)})

In [ ]:
# Sanity check: all model inputs must be numeric before training (fails early with a clear message)
assert not any(Xtr.dtypes.map(lambda t: not pd.api.types.is_numeric_dtype(t))), "Non-numeric column left in Xtr"
assert Xtr.shape[1] == Xva.shape[1] == Xte.shape[1], "Train/val/test feature mismatch"
print("OK: Xtr", Xtr.shape, "| Xva", Xva.shape, "| Xte", Xte.shape, "| attack rate (train):", round(ytr.mean(), 3))

## 4. Evaluation toolbox
Reusable functions used by **both** engines:
* `sweep()` – evaluates *every* candidate threshold (precision, recall, FPR, alert rate, F-beta, SOC cost).
* `pick_operating_points()` – 4 strategies for choosing a threshold.
* `triage_thresholds()` / `tier_report()` – the 3-tier SOC policy.
* `report()` – metrics at a fixed threshold.

In [ ]:
def sweep(y, s, beta=F_BETA, n_grid=1500):
    """Vectorised threshold sweep. Flag = score >= threshold."""
    y, s = np.asarray(y), np.asarray(s)
    thr = np.unique(np.quantile(s, np.linspace(0, 1, n_grid)))
    sp, sn = np.sort(s[y == 1]), np.sort(s[y == 0])
    tp = len(sp) - np.searchsorted(sp, thr, side="left")
    fp = len(sn) - np.searchsorted(sn, thr, side="left")
    fn, tn = len(sp) - tp, len(sn) - fp
    prec = np.divide(tp, tp + fp, out=np.ones(len(thr)), where=(tp + fp) > 0)
    rec = tp / len(sp)
    b2 = beta ** 2
    den = b2 * prec + rec
    fb = np.divide((1 + b2) * prec * rec, den, out=np.zeros(len(thr)), where=den > 0)
    return pd.DataFrame(dict(threshold=thr, precision=prec, recall=rec, fpr=fp / len(sn), f_beta=fb,
                             flagged_rate=(tp + fp) / len(s), TP=tp, FP=fp, FN=fn, TN=tn,
                             cost=COST_FP * fp + COST_FN * fn))

def pick_operating_points(sw, precision_floor=PRECISION_FLOOR, budget=ALERT_BUDGET):
    """Four ways to choose a threshold -> returns a DataFrame (one row per strategy)."""
    rows = {}
    rows["min_cost (SOC economics)"] = sw.loc[sw["cost"].idxmin()]
    rows[f"max_F{F_BETA:g} (recall-weighted)"] = sw.loc[sw["f_beta"].idxmax()]
    ok = sw[sw["precision"] >= precision_floor]
    rows[f"precision>={precision_floor:.0%} (max recall)"] = ok.loc[ok["recall"].idxmax()] if len(ok) else sw.loc[sw["precision"].idxmax()]
    ok = sw[sw["flagged_rate"] <= budget]
    rows[f"alert budget<={budget:.0%} (max recall)"] = ok.loc[ok["recall"].idxmax()] if len(ok) else sw.iloc[-1]
    return pd.DataFrame(rows).T

def report(y, s, thr, name=""):
    pred = (np.asarray(s) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return dict(model=name, threshold=thr, precision=prec, recall=rec,
                f1=2 * prec * rec / max(prec + rec, 1e-12), FPR=fp / max(fp + tn, 1),
                TP=tp, FP=fp, FN=fn, TN=tn, PR_AUC=average_precision_score(y, s),
                ROC_AUC=roc_auc_score(y, s), SOC_cost=COST_FP * fp + COST_FN * fn)

def triage_thresholds(y, p, max_missed_frac=MAX_MISSED_FRAC, min_precision=MIN_PRECISION_ESC):
    """t_lo: below it we auto-close (hides <= max_missed_frac of attacks).  t_hi: above it we auto-escalate (precision >= min_precision)."""
    y, p = np.asarray(y), np.asarray(p)
    order = np.argsort(p); ps = p[order]
    cum = np.cumsum(y[order]) / y.sum()
    i = np.searchsorted(cum, max_missed_frac, side="right")
    t_lo = ps[i - 1] if i > 0 else 0.0
    prec, rec, thr = precision_recall_curve(y, p)
    ok = np.where(prec[:-1] >= min_precision)[0]
    t_hi = thr[ok.min()] if len(ok) else 1.0
    return min(t_lo, t_hi), t_hi

def tier_report(y, p, t_lo, t_hi, label=""):
    y, p = np.asarray(y), np.asarray(p)
    closed, esc = p < t_lo, p >= t_hi
    review = ~closed & ~esc
    n_attacks, n_benign = y.sum(), (1 - y).sum()
    out = {
        "auto-closed events": int(closed.sum()),
        "  attacks hidden in auto-closed (missed)": int(y[closed].sum()),
        "auto-escalated events": int(esc.sum()),
        "  false positives in auto-escalated": int((1 - y[esc]).sum()),
        "analyst-review queue": int(review.sum()),
        "  attacks in queue": int(y[review].sum()),
        "% events needing a human": 100 * review.mean(),
        "% benign events never shown to a human": 100 * ((1 - y)[~review].sum() / n_benign),
        "% attacks caught (escalated + queued)": 100 * (y[~closed].sum() / n_attacks)}
    return pd.Series(out, name=label)

def plot_pr(ax, y, scores: dict):
    for name, s in scores.items():
        p, r, _ = precision_recall_curve(y, s)
        ax.plot(r, p, label=f"{name} (AP={average_precision_score(y, s):.3f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend(loc="lower left"); ax.set_title("Precision-Recall")

## 5. Supervised engine — automated alert triage
Goal: **drastically reduce false positives** while keeping missed attacks near zero.

We train **LightGBM** (main model, early-stopped on validation) and a **Random Forest** (baseline).

In [ ]:
t0 = time.time()
lgbm = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=63, min_child_samples=30,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                          random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
lgbm.fit(Xtr, ytr, eval_set=[(Xva, yva)],
         callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
print(f"LightGBM trained: {lgbm.n_estimators_} trees used, best iteration = {lgbm.best_iteration_}  ({time.time()-t0:.0f}s)")

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample",
                            n_jobs=-1, random_state=RANDOM_STATE).fit(Xtr, ytr)
print(f"RandomForest trained ({time.time()-t0:.0f}s)")

In [ ]:
p_va = {"LightGBM": lgbm.predict_proba(Xva)[:, 1], "RandomForest": rf.predict_proba(Xva)[:, 1]}
print({k: f"AP={average_precision_score(yva, v):.4f}" for k, v in p_va.items()})

### 5.1 Tune the supervised decision policy on **validation**
First the classic single threshold, then the 3-tier SOC policy.

In [ ]:
sw_lgb = sweep(yva, p_va["LightGBM"])
ops_lgb = pick_operating_points(sw_lgb)
display(ops_lgb[["threshold", "precision", "recall", "fpr", "flagged_rate", "FP", "FN", "cost"]])

T_SUP = float(ops_lgb.iloc[0]["threshold"])           # min-cost threshold = default SOC choice
T_LO, T_HI = triage_thresholds(yva, p_va["LightGBM"])
print(f"\nSupervised single threshold (min-cost): {T_SUP:.4f}")
print(f"3-tier policy: auto-close if p < {T_LO:.4f} | analyst review if {T_LO:.4f} <= p < {T_HI:.4f} | auto-escalate if p >= {T_HI:.4f}")
display(tier_report(yva, p_va["LightGBM"], T_LO, T_HI, "validation").to_frame())

## 6. Unsupervised engine — behavioural anomaly detection
Both models are trained **only on normal behaviour** (no attack labels), so they model "what normal looks like" and flag deviations.
This is how you catch **novel / polymorphic** activity (UNSW) or **unusual user behaviour** (CERT insider threats).

* **Isolation Forest** — attacks are "easy to isolate" → short tree paths.
* **Autoencoder** — compresses normal data through a bottleneck; anomalies **reconstruct badly** (high error).
  Implemented with scikit-learn's `MLPRegressor` (input → input) so no deep-learning framework is required.

In [ ]:
normal_idx = np.where(ytr == 0)[0]

# ---------------- Isolation Forest ----------------
t0 = time.time()
iso = IsolationForest(n_estimators=400, max_samples=512, contamination="auto",
                      random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr.iloc[normal_idx])
iso_score = lambda X: -iso.score_samples(X)                  # higher = more anomalous
print(f"IsolationForest trained ({time.time()-t0:.0f}s)")

# ---------------- Autoencoder ----------------
t0 = time.time()
rng = np.random.RandomState(RANDOM_STATE)
ae_idx = rng.choice(normal_idx, size=min(AE_MAX_ROWS, len(normal_idx)), replace=False)
Z = to_ae(tr_df.iloc[ae_idx])
d = Z.shape[1]
ae = MLPRegressor(hidden_layer_sizes=(128, 64, 16, 64, 128), activation="relu", alpha=1e-5,
                  learning_rate_init=1e-3, batch_size=256, max_iter=60, early_stopping=True,
                  validation_fraction=0.1, n_iter_no_change=6, random_state=RANDOM_STATE)
ae.fit(Z, Z)                                                  # reconstruct the input
ae_score = lambda df: ((to_ae(df) - ae.predict(to_ae(df))) ** 2).mean(axis=1)   # reconstruction MSE
print(f"Autoencoder trained on {len(Z):,} normal rows, {d} inputs -> bottleneck 16  ({time.time()-t0:.0f}s)")

s_va = {"IsolationForest": iso_score(Xva), "Autoencoder": ae_score(va_df)}
print({k: f"AP={average_precision_score(yva, v):.4f} | ROC-AUC={roc_auc_score(yva, v):.4f}" for k, v in s_va.items()})

plt.figure(figsize=(7, 4))
plt.plot(ae.loss_curve_); plt.title("Autoencoder training loss"); plt.xlabel("epoch"); plt.ylabel("loss"); show()

## 7. 🎯 Optimization task — anomaly-threshold tuning for a high-stakes SOC
An anomaly score is useless until it becomes a **yes/no alert**. The threshold controls the trade-off:

* **Too low** → alert fatigue (analysts drown in false positives, real attacks get ignored).
* **Too high** → missed attacks (breach).

We compare four strategies **on validation data only**:

| Strategy | Question it answers |
|---|---|
| `min_cost` | Which threshold minimises `COST_FP·FP + COST_FN·FN`? |
| `max_F2` | Best balance when recall matters 2× more than precision |
| `precision floor` | Highest recall while keeping analysts' time well spent (≥ 90 % true alerts) |
| `alert budget` | Highest recall given a fixed daily capacity (e.g., 5 % of events) |

In [ ]:
sw = {name: sweep(yva, s) for name, s in s_va.items()}
ops = {name: pick_operating_points(sw[name]) for name in sw}
for name in ops:
    print(f"\n=== {name}: operating points (validation) ===")
    display(ops[name][["threshold", "precision", "recall", "fpr", "flagged_rate", "FP", "FN", "cost"]])

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for j, name in enumerate(sw):
    s = sw[name]
    ax = axes[0, j]
    ax.plot(s["threshold"], s["precision"], label="precision"); ax.plot(s["threshold"], s["recall"], label="recall")
    ax.plot(s["threshold"], s["f_beta"], label=f"F{F_BETA:g}", ls="--")
    ax.axvline(ops[name].iloc[0]["threshold"], color="k", ls=":", label="min-cost thr")
    ax.set_title(f"{name}: precision / recall vs threshold"); ax.set_xlabel("threshold"); ax.legend()
    ax = axes[1, j]
    ax.plot(s["threshold"], s["cost"]); ax.axvline(ops[name].iloc[0]["threshold"], color="k", ls=":")
    ax.set_title(f"{name}: SOC cost (FP×{COST_FP} + FN×{COST_FN})"); ax.set_xlabel("threshold"); ax.set_ylabel("cost")
plt.tight_layout(); show()

### 7.1 Label-free fallback (for real deployments)
In production you often have **no labels** to tune on. A defensible alternative is to set the threshold at a high percentile of the **normal-only** score distribution, i.e. *"I accept at most X false alerts per 1,000 benign events"*.

In [ ]:
FPR_TARGET = 0.01       # accept ~1% false alerts on benign traffic
label_free = {name: float(np.quantile(s[yva == 0], 1 - FPR_TARGET)) for name, s in s_va.items()}
for name, t in label_free.items():
    r = report(yva, s_va[name], t, name)
    print(f"{name:16s} thr={t:.5f}  precision={r['precision']:.3f}  recall={r['recall']:.3f}  FPR={r['FPR']:.3f}")

## 8. Hybrid risk score — combine both engines
The supervised model is great at **known** attacks; the anomaly detectors help with **unknown** ones.
We convert anomaly scores to percentile ranks vs. normal validation data (so they're comparable) and blend:

`risk = w · P(attack | supervised)  +  (1 − w) · max(IF_rank, AE_rank)`

`w` is chosen on validation by PR-AUC.

In [ ]:
def make_rank(ref_scores):
    ref = np.sort(ref_scores)
    return lambda s: np.searchsorted(ref, s, side="right") / len(ref)

rank_if = make_rank(s_va["IsolationForest"][yva == 0])
rank_ae = make_rank(s_va["Autoencoder"][yva == 0])

def hybrid(p_sup, s_if, s_ae, w):
    return w * p_sup + (1 - w) * np.maximum(rank_if(s_if), rank_ae(s_ae))

grid = np.round(np.arange(0.0, 1.01, 0.1), 2)
ap = [average_precision_score(yva, hybrid(p_va["LightGBM"], s_va["IsolationForest"], s_va["Autoencoder"], w)) for w in grid]
W = float(grid[int(np.argmax(ap))])
plt.plot(grid, ap, marker="o"); plt.axvline(W, color="k", ls=":"); plt.xlabel("w (weight on supervised)"); plt.ylabel("validation PR-AUC")
plt.title(f"Hybrid weight search -> w = {W}"); show()

hy_va = hybrid(p_va["LightGBM"], s_va["IsolationForest"], s_va["Autoencoder"], W)
ops_hy = pick_operating_points(sweep(yva, hy_va))
display(ops_hy[["threshold", "precision", "recall", "fpr", "flagged_rate", "FP", "FN", "cost"]])

## 9. 🧪 Final evaluation on the held-out **testing-set**
Thresholds below were all chosen on validation. Nothing is re-tuned on test.

In [ ]:
# ---- score the test set with every model ----
p_te = {"LightGBM": lgbm.predict_proba(Xte)[:, 1], "RandomForest": rf.predict_proba(Xte)[:, 1]}
s_te = {"IsolationForest": iso_score(Xte), "Autoencoder": ae_score(te_df)}
hy_te = hybrid(p_te["LightGBM"], s_te["IsolationForest"], s_te["Autoencoder"], W)

# ---- thresholds chosen on validation (min-cost strategy) ----
thr = {"LightGBM": T_SUP,
       "RandomForest": float(pick_operating_points(sweep(yva, p_va["RandomForest"])).iloc[0]["threshold"]),
       "IsolationForest": float(ops["IsolationForest"].iloc[0]["threshold"]),
       "Autoencoder": float(ops["Autoencoder"].iloc[0]["threshold"]),
       "Hybrid": float(ops_hy.iloc[0]["threshold"])}
all_te = {**p_te, **s_te, "Hybrid": hy_te}

rows = [report(yte, all_te[m], thr[m], m) for m in all_te]
# naive baselines for context
rows.append(report(yte, p_te["LightGBM"], 0.5, "LightGBM @ default 0.5"))
res = pd.DataFrame(rows).set_index("model")
display(res[["threshold", "precision", "recall", "f1", "FPR", "FP", "FN", "PR_AUC", "ROC_AUC", "SOC_cost"]])

# ---- False-positive reduction vs 'alert on everything' ----
n_benign = int((yte == 0).sum())
fpr_tbl = pd.DataFrame({"false alerts": res["FP"], "% of benign events falsely flagged": 100 * res["FP"] / n_benign,
                        "FP reduction vs alert-on-everything (%)": 100 * (1 - res["FP"] / n_benign)})
display(fpr_tbl)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
plot_pr(ax[0], yte, all_te)
from sklearn.metrics import roc_curve
for n, s in all_te.items():
    f, t, _ = roc_curve(yte, s); ax[1].plot(f, t, label=f"{n} ({roc_auc_score(yte, s):.3f})")
ax[1].plot([0, 1], [0, 1], "k:"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].legend(); ax[1].set_title("ROC")
plt.tight_layout(); show()

### 9.1 3-tier SOC triage on the test set
How much analyst workload does the supervised engine remove, and how many attacks slip through?

In [ ]:
display(tier_report(yte, p_te["LightGBM"], T_LO, T_HI, "test (3-tier policy)").to_frame())

# confusion matrices at the chosen thresholds
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, m in zip(axes, ["LightGBM", "Autoencoder", "Hybrid"]):
    cm = confusion_matrix(yte, (all_te[m] >= thr[m]).astype(int), labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm): ax.text(j, i, f"{v:,}", ha="center", va="center", color="black")
    ax.set_xticks([0, 1], ["pred benign", "pred attack"]); ax.set_yticks([0, 1], ["benign", "attack"]); ax.set_title(m); ax.grid(False)
plt.tight_layout(); show()

## 10. Which attack families are caught? (per-category recall)
Only available when the dataset has a category column (UNSW-NB15: `attack_cat`).

In [ ]:
def cat_recall(df, scores, thr_):
    a = df[df[LABEL] == 1].copy()
    a["hit"] = (np.asarray(scores)[a.index] >= thr_).astype(int)
    return a.groupby(CAT_COL)["hit"].mean()

if CAT_COL:
    tbl = pd.DataFrame({m: cat_recall(te_df, all_te[m], thr[m]) for m in ["LightGBM", "RandomForest", "IsolationForest", "Autoencoder", "Hybrid"]})
    tbl["n_test"] = te_df[te_df[LABEL] == 1][CAT_COL].value_counts()
    display((tbl.sort_values("n_test", ascending=False)))
    tbl.drop(columns="n_test").plot.bar(figsize=(11, 4), title="Recall per attack family (test set, validation-tuned thresholds)")
    plt.ylabel("recall"); plt.tight_layout(); show()
else:
    print("No category column for this dataset - skipping.")

## 11. 🧬 Zero-day simulation
Remove entire attack families from **training**, then check who still catches them in the test set.
The supervised model has *never seen* these families; the unsupervised engines never see **any** attack.
This quantifies why you want **both** engines.

In [ ]:
HOLDOUT = ["Backdoor", "Shellcode", "Worms", "Analysis"]      # <-- edit: families to hide from the supervised model

if CAT_COL:
    keep_tr = ~tr_df[CAT_COL].isin(HOLDOUT).values
    keep_va = ~va_df[CAT_COL].isin(HOLDOUT).values
    lgb_zd = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=RANDOM_STATE,
                                n_jobs=-1, verbose=-1).fit(Xtr[keep_tr], ytr[keep_tr])
    # thresholds tuned WITHOUT the held-out families (they are 'unknown' to the SOC)
    t_zd = float(pick_operating_points(sweep(yva[keep_va], lgb_zd.predict_proba(Xva[keep_va])[:, 1])).iloc[0]["threshold"])
    t_if = float(pick_operating_points(sweep(yva[keep_va], s_va["IsolationForest"][keep_va])).iloc[0]["threshold"])
    t_ae = float(pick_operating_points(sweep(yva[keep_va], s_va["Autoencoder"][keep_va])).iloc[0]["threshold"])

    sc = {"LightGBM (blind to these)": (lgb_zd.predict_proba(Xte)[:, 1], t_zd),
          "IsolationForest": (s_te["IsolationForest"], t_if), "Autoencoder": (s_te["Autoencoder"], t_ae)}
    hold_df = te_df[te_df[CAT_COL].isin(HOLDOUT)]
    zd = pd.DataFrame({n: hold_df.assign(hit=(s[hold_df.index] >= t).astype(int)).groupby(CAT_COL)["hit"].mean()
                       for n, (s, t) in sc.items()})
    zd["n_test"] = hold_df[CAT_COL].value_counts()
    display(zd)
    print("Recall on families unseen in training. Where the supervised model is blind, the anomaly engines are the safety net.")
else:
    print("No category column - skipping zero-day simulation (for CERT, hold out whole insider scenarios instead).")

## 12. 🔍 Explainability — *why* did this alert fire?
Analysts trust (and act on) alerts they can understand. LightGBM can return per-feature contributions (SHAP-style) with no extra library.

In [ ]:
imp = pd.Series(lgbm.booster_.feature_importance("gain"), index=Xtr.columns).sort_values(ascending=False)
imp.head(20)[::-1].plot.barh(figsize=(7, 6), title="Top-20 features (gain)"); plt.tight_layout(); show()

def explain_alert(i, X=Xte, df=te_df, top=6):
    """Top features pushing event i towards 'attack' (+) or 'benign' (-)."""
    contrib = lgbm.booster_.predict(X.iloc[[i]], pred_contrib=True)[0][:-1]
    s = pd.Series(contrib, index=X.columns)
    top_idx = s.abs().sort_values(ascending=False).index[:top]
    out = pd.DataFrame({"value": X.iloc[i][top_idx], "contribution": s[top_idx]})
    print(f"Event #{i} | P(attack)={p_te['LightGBM'][i]:.3f} | true label={yte[i]}" + (f" | category={df.iloc[i][CAT_COL]}" if CAT_COL else ""))
    return out

# explain the highest-risk true attack and a false positive (if any)
i_tp = int(np.argmax(np.where(yte == 1, p_te["LightGBM"], -1)))
display(explain_alert(i_tp))
fp_idx = np.where((yte == 0) & (p_te["LightGBM"] >= thr["LightGBM"]))[0]
if len(fp_idx): display(explain_alert(int(fp_idx[0])))

## 13. Save the trained SOC engine

In [ ]:
OUT = Path("soc_artifacts"); OUT.mkdir(exist_ok=True)
joblib.dump({"dataset": DATASET, "features": FEATURES, "sup_pre": sup_pre, "ae_pre": ae_pre,
             "lgbm": lgbm, "rf": rf, "iso": iso, "ae": ae,
             "thresholds": {**thr, "tier_lo": T_LO, "tier_hi": T_HI}, "hybrid_w": W}, OUT / f"soc_engine_{DATASET}.joblib")
res.to_csv(OUT / f"test_results_{DATASET}.csv")
print("Saved to", OUT.resolve())

## 14. 📌 Interpretation guide (fill in with your own numbers)

1. **Supervised triage:** compare `FP` for *LightGBM @ 0.5* vs *LightGBM (tuned)* — the tuned policy should shrink false alerts while keeping `FN` tiny. Report the 3-tier table: % of events that need a human.
2. **Unsupervised:** Isolation Forest vs Autoencoder — which has higher PR-AUC? Which family recall is higher? Autoencoders usually shine with correlated numeric features; Isolation Forest is faster and more robust.
3. **Threshold optimisation:** show how the optimum moves when you change `COST_FN` (try 10, 50, 200). A bigger cost of a miss pushes the threshold *down* (more alerts, more recall).
4. **Zero-day test:** show that LightGBM's recall collapses on hidden families while the anomaly engines retain some detection → justification for the hybrid system.
5. **Limitations:** UNSW-NB15 is synthetic and its train/test mix differs; supervised scores on it are optimistic compared with real networks; thresholds should be re-tuned and monitored for drift.

---
## 🔁 How to switch to the CERT insider-threat dataset
1. Download CERT r4.2 (or r5.x) and place `logon.csv, device.csv, file.csv, email.csv, http.csv` in `CERT_DIR`, and `answers/insiders.csv` in `CERT_ANSWERS`.
2. In **Section 1** set `DATASET = "CERT"` (and `CERT_VERSION` to match `insiders.csv`).
3. *Restart & run all.* Everything downstream (features, splits, LightGBM/RF, Isolation Forest, Autoencoder, threshold tuning, hybrid) works unchanged because the adapter returns the same `(train, test)` contract.
4. Tips for CERT: attacks are **extremely rare** (≪ 1 % of user-days) → raise `COST_FN`, use `ALERT_BUDGET` ≈ 0.5–1 %, and prefer PR-AUC over accuracy. Section 11 (zero-day) is skipped because there is no category column; add an `insider scenario` column in `load_cert` and set `category_col` in `DATASETS` to enable it.
5. The CERT adapter streams the big `http.csv` in chunks; first load can take several minutes — consider caching the user-day table with `daily.to_parquet(...)`.